In [ ]:
import spacy
#Load in the base spacy model
#nlp_sents = spacy.load('en_core_web_lg', exclude = ["parser", "textcat", "ner"])
#nlp_sents.enable_pipe("senter")
#Load up the newly improved text classification model for spacy 3.0


path = '../training/model-best'
#path = '../training/model-best'
nlp_textcat = spacy.load(path) 

def textcat_crypto(text, threshold):
    doc = nlp_textcat(text)
    return "RELEVANT" if doc.cats['RELEVANT'] >= threshold else "IRRELEVANT"


In [ ]:
from snorkel.labeling import labeling_function

symbols = ['🚨','⚠️','🚩','❌', '👀'] 

#CommunityAlert 🚨

warnings = ['Be careful', 'watch out']

# Hashtag Patterns
hashtag_regex = '#[a-zA-Z]+'

twitter_hashtag = '([#＃]+)([0-9A-Z_]*[A-Z_]+[a-z0-9_üÀ-ÖØ-öø-ÿ]*)'

# Scam Hashtags

scam_regex = '#Scam.{1,6}'
scam_regex2 = '#Scam|scam|SCAM.{1,6}\S'

# Pattern 2

ether_address = '0x[a-fA-F0-9]{40}'

# Pattern number 3
# #Scamalert and #Cryptocurrency

# Pattern number 4 
# Spacy model contains MONEY

# Pattern number 5
# use huggingface to detect anger for RELEVANT
# Use joy and optimism for IRRELEVANT
# https://huggingface.co/cardiffnlp/twitter-roberta-base-emotion


import re


@labeling_function()
def regex_scam(x):
    return "RELEVANT" if re.search(r"#Scam|scam|SCAM.{1,6}\S", x.text, flags=re.I) else "IGNORE"


from snorkel.preprocess.nlp import SpacyPreprocessor

# The SpacyPreprocessor parses the text in text_field and
# stores the new enriched representation in doc_field
spacy = SpacyPreprocessor(text_field="text", doc_field="doc", memoize=True)


# If Sentiment is negative and it includes MONEY entity then...
@labeling_function(pre=[spacy])
def has_person(x):
    """Ham comments mention specific people and are short."""
    if len(x.doc) < 20 and any([ent.label_ == "MONEY" for ent in x.doc.ents]):
        return "RELEVANT"
    else:
        return "IRRELEVANT"

from snorkel.labeling.lf.nlp import nlp_labeling_function

@nlp_labeling_function()
def has_person_nlp(x):
    """Ham comments mention specific people and are short."""
    if len(x.doc) < 20 and any([ent.label_ == "MONEY" for ent in x.doc.ents]):
        return "RELEVANT"
    else:
        return "IRRELEVANT"

In [ ]:
from snorkel.preprocess import preprocessor
from textblob import TextBlob
import spacy

@preprocessor(memoize=True)
# path = '../training/model-best'
def textcat_crypto(path, x):
    nlp_textcat = spacy.load(path) 
    doc = nlp_textcat(x)
    return doc.cats


@labeling_function(pre=[textcat_crypto])
# path = '../training/model-best'
def textcat_crypto(x):    
    return "RELEVANT" if x['RELEVANT'] >= 0.70 else "IRRELEVANT"



import spacy

path = '../patterns/patterns.jsonl'

def entity_ruler(text, path):
    labels = ['CYBER', 'MONEY', 'SCAMS']
    nlp_sents = spacy.load('en_core_web_sm', exclude = ["parser", "textcat"])
    nlp_sents.add_pipe("entity_ruler").from_disk(path)
    text = text.lower()
    doc = nlp_sents(text)
    res = any([(ent.text) for ent in doc.ents if ent.label_ in labels])
    return 'RELEVANT' if res is True else 'IRRELEVANT'

In [ ]:
# For NER Modeling use Weak Supervision:
# Conduct more twitter queries for #ScamAlert
# Build Gazeteer of Cryptonames and Crypto tokens
# Crypto-related terms
# Scam-related terms
# 

In [ ]:

#TODO Extract sentences from the Entity Ruler Patterns from File. !!!!!!!!!!!!!!!!!!!!

import spacy
#nlp = spacy.load("en_core_web_lg")

path = '../patterns/patterns.jsonl'

nlp_sents = spacy.load('en_core_web_lg', exclude = ["parser", "textcat"])
nlp_sents.enable_pipe("senter")
new_ruler = nlp_sents.add_pipe("entity_ruler").from_disk(path)

doc = nlp_sents("The drugs are named: heroin, FRAUDSTERS, malware, cocaine, and morphine. BTC We scammed these guys and did a rug pull in Armenia")
print([(ent.text, ent.label_) for ent in doc.ents])


In [ ]:
# Identify if terms are present in any particular Tweet
# CRYPTO CYBER SCAM
import spacy

path = '../patterns/patterns.jsonl'

def entity_ruler(text, path):
    labels = ['CYBER', 'SCAMS']
    nlp_sents = spacy.load('en_core_web_sm', exclude = ["parser", "textcat"])
    nlp_sents.add_pipe("entity_ruler").from_disk(path)
    text = text.lower()
    doc = nlp_sents(text)
    res = any([(ent.text) for ent in doc.ents if ent.label_ in labels])
    return 'RELEVANT' if res is True else 'IRRELEVANT'

In [ ]:
# Use this github for the crypto currency media analysis for key terms

"https://github.com/matthewkmoore/CryptoCurrency-Media-Analysis"

In [ ]:
"""free giveaway
presale
PRESALE
#NFTGiveaway🎁
giving
#GIVEAWAY
Follow me
giveaways
#moonshot
#Giveaways
Airdrop
moon
rocket

follow
followers
support
supporters
subscribe
"""

In [ ]:
"""(Giveaway|GIVEAWAY|giveaway|giving|Giving){1}
(follow|FOLLOW|Follow){1}
(presale|Presale|PRESALE){1}
(subscribe|Subscribe|SUBSCRIBE){1}
(rocket|moon|Airdrop|AIRDROP){1}"""

In [ ]:
train = "../corpus/data/twitter_training_data.csv"
# test = "../corpus/data/twitter_test_data.csv"

import pandas as pd

df_train = pd.read_csv(train)

In [ ]:
df_train.head()

In [ ]:
df_train["label"] = df_train["text"].apply(lambda x: entity_ruler(x, path))

In [ ]:
train = "../corpus/data/twitter_training_data.csv"
# test = "../corpus/data/twitter_test_data.csv"

import pandas as pd

df_train = pd.read_csv(train)
df_train = df_train.drop_duplicates(subset='text')
#df_train = df_train.sample(frac=0.20).reset_index(drop=True)

In [ ]:
len(df_train)

In [ ]:
df_test = df_train.sample(frac=0.05).reset_index(drop=True)

In [ ]:
len(df_test)

In [ ]:
len(df_test)

In [ ]:
df_test["label"] = df_test["text"].apply(lambda x: entity_ruler(x, path))

In [ ]:
len(df_test)

In [ ]:
df_test = df_test.loc[df_test['label'] == "RELEVANT"]

In [ ]:
# df_test.to_csv("../corpus/data/filtered_training_data.csv", index=False, encoding="utf-8")

import pandas as pd
df_test = pd.read_csv("../corpus/data/reddit/crypto_data.csv", engine='python')

In [ ]:
import spacy
path = '../training/model-best'
#path = '../training/model-best'
nlp_textcat = spacy.load(path) 

def textcat_crypto(text, threshold):
    doc = nlp_textcat(text)
    return "RELEVANT" if doc.cats['RELEVANT'] >= threshold else "IRRELEVANT"

In [ ]:
#df1 = df_test.loc[0:1000]

#df2 = df_test.iloc[1000:10000]
df4 = df_test.iloc[30000:]
#df1 = df_test.loc[0:1000]
#df1 = df_test.loc[0:1000]

In [ ]:
df4["labels"] = df4["text"].apply(lambda x: textcat_crypto(x, 0.75))

In [ ]:
frames = [df1, df2, df3, df4]
total = pd.concat(frames)

In [ ]:
total = total[['text', 'labels']]

In [ ]:
total = total.sort_values(by=['labels'], ascending=False)

In [ ]:
df_test.sort_index(ascending=False, axis=1)

In [ ]:
total = total.loc[total['labels'] == "RELEVANT"]
print(len(total))

In [ ]:
total

In [ ]:
import random
totals = list(total['text'])

totals.shuffle()


In [ ]:
random.shuffle(totals)

In [ ]:
df2 = df_test[['text', 'labels']]

In [ ]:
# df_re = df_test.loc[df_test['label'] == "RELEVANT"]


df2 = df2.sort_values(by=['labels'], ascending=False)

In [ ]:
df2.to_csv("../corpus/data/twitter_annotation_data.csv", index=False, encoding="utf-8")

In [ ]:
import re, ftfy
def preProcess(text):
    """
    Do a little bit of data cleaning with the help of Unidecode and Regex.
    Things like casing, extra spaces, quotes and new lines can be ignored.
    """    
    text = ftfy.fix_text(text)
    text = re.sub(r"^[^a-zA-Z]+", '', text)
    text = re.sub('  +', ' ', text)
    text = re.sub('\n', ' ', text)
    text = text.strip().strip('"').strip("'").strip()
    # If data is missing, indicate that by setting the value to `None`
    return "RELEVANT" if len(text) >= 150 else "IRRELEVANT"


In [ ]:
from snorkel.labeling.lf import labeling_function
from snorkelflow.studio import NodeDataset
terms_dict = [
    (['equal', 'principal'], 'POSITIVE'),
    (['less', 'more'], 'NEGATIVE')
]

def make_keyword_lf(keywords, label_str):
    """Generates an LF given keywords / label_str params."""

    lf_name = f"generated-{label_str}-lf"
    joined_keywords = "|".join(keywords)
    regex = re.compile(rf"\b(?:{joined_keywords})\b")

    @labeling_function(name=lf_name, resources=dict(regex=regex, label_str=label_str))
    def generated_lf(x, regex, label_str):
        import re
        if re.match(regex, x.span_preview):
            return label_str
        return "UNKNOWN"

    return generated_lf
    
# APP_NAME = <insert your application name here>
node_dataset = NodeDataset(application_name=APP_NAME)
for keywords, label_str in terms_dict:
    node_dataset.save(make_keyword_lf(keywords, label_str), label_str=label_str)

In [ ]:
from snorkelflow.models.model_registry import get_model_from_config
model_config = [YOUR MODEL CONFIG]
model = get_model_from_config(model_config)

In [ ]:
from snorkelflow.models.model_configs import DISTILBERT_CLASSIFICATION_CONFIG

# Create a model with certain parameters and specify fields to train over
model_config = copy.deepcopy(DISTILBERT_CLASSIFICATION_CONFIG)
print(model_config)

In [ ]:
import bz2
import json
import srsly

output = '../corpus/data/test.jsonl'
path = '../corpus/data/raw/twitter-stream.json.bz2'

def json_writer(path):
    lines = []
    with bz2.open(path, "rt") as bzinput:
        for i, line in enumerate(bzinput):
            #if i == 10: break
            tweets = json.loads(line)
            lines.append(tweets)
        # if adding multiple paths then add a yield generator
        #yield lines
        return lines

test = json_writer(path)

#srsly.write_jsonl('../corpus/data/test.jsonl', test)

data = []
for t in test:
    try:
        if t['lang'] == 'en':
            data.append({
                'date_created': t['created_at'],
                'text': t['text'],
                'ID': t['id_str'],
                })
        srsly.write_jsonl('../corpus/data/test.jsonl', data)
    except KeyError as E:
        continue

In [ ]:
len(data)

In [ ]:
records = []

for d in totals:
    records.append({
        "text": d
    })

In [ ]:
import srsly
srsly.write_jsonl("../corpus/data/new_annotation_data.jsonl", records)